# Performance drought metrics

In [ ]:
# general
from scipy.stats import spearmanr
from pathlib import Path
import pandas as pd

# plotting
import plotly.graph_objects as go
from   plotly.subplots import make_subplots
from sklearn.preprocessing import MinMaxScaler

from src.utils import load_config
from src.plotting import (
    base_layout, FONT_FAMILY, PANEL_LABEL_COLOR, DEFAULT_MARGIN,
    SEQUENTIAL_COLORSCALE, REFERENCE_LINE_COLOR, save_figure,
)
cfg = load_config()

## Data

In [ ]:
# metrics 
metrics_droughts    = pd.read_csv("data/hydro/performance_lowflows.csv", index_col=0)
metrics_split = pd.read_csv("data/hydro/performance_metrics.csv", index_col=0).split,

# andeanGC metadata
andeanGC_metadata = pd.read_csv(Path(cfg["dir"]["andeangc"]) / "dataset/AndeanGC_metadata.csv", index_col=0)
andeanGC_metadata = andeanGC_metadata.join(metrics_droughts, how="left").dropna(axis = 0)
andeanGC_metadata = andeanGC_metadata.join(metrics_split, how="left").dropna(axis =0)

## Plot

In [ ]:
# Create subplots
fig = make_subplots(rows=2, cols=3, horizontal_spacing=0.08, vertical_spacing=0.11)

# Define styling parameters
colorscale = SEQUENTIAL_COLORSCALE
scaler = MinMaxScaler(feature_range=(7, 25))
line_train = "grey"
line_validation = "#0c0c0c"
opacity = 0.95

# Helper function to add scatter plots
def add_scatter_plot(x_data, y_data, data_type, row, col, show_colorbar=False):
    dataset = andeanGC_metadata[andeanGC_metadata.split == data_type]
    line_color = line_train if data_type == 'train' else line_validation
    
    fig.add_trace(go.Scatter(
        x=dataset[x_data], y=dataset[y_data],
        mode='markers', name=data_type.capitalize(), opacity=opacity, showlegend=False,
        marker=dict(
            size=scaler.fit_transform(dataset[['basin_area']]).flatten(),
            color=dataset['glacier_area_RGI60'], 
            cmax=25, cmin=0, showscale=show_colorbar,
            colorbar=dict(title="Glacier<br>area", x=1.01, thickness=18, tickwidth=1, ticksuffix="%"),
            colorscale=colorscale, line_width=1, line_color=line_color
        )
    ), row=row, col=col)

# Helper function to add 1:1 reference line
def add_reference_line(range_vals, row, col):
    fig.add_trace(go.Scatter(
        x=range_vals, y=range_vals, mode='lines',
        line=dict(color=REFERENCE_LINE_COLOR, dash='dot'), 
        showlegend=False
    ), row=row, col=col)

# Plot configurations
plots = [
    {'x': 'Q90_obs', 'y': 'Q90_sim', 'range': [-0.5, 6], 'dtick': 2, 'row': 1, 'col': 1,
     'xlabel': 'Q<sub>90</sub> (mm d<sup>-1</sup>) <span style="color:lightgrey">[obs]</span>', 
     'ylabel': 'Q<sub>90</sub> (mm d<sup>-1</sup>) <span style="color:lightgrey">[sim]</span>'},
    {'x': 'min_30day_obs', 'y': 'min_30day_sim', 'range': [-0.5, 5], 'dtick': 1, 'row': 1, 'col': 2,
     'xlabel': 'Min 30-day flow (mm d<sup>-1</sup>) <span style="color:lightgrey">[obs]</span>', 
     'ylabel': 'Min 30-day flow (mm d<sup>-1</sup>) <span style="color:lightgrey">[sim]</span>'},
    {'x': 'doy_min_obs', 'y': 'doy_min_sim', 'range': [60, 330], 'dtick': 50, 'row': 2, 'col': 1,
     'xlabel': 'DOY min. flow (julian day) <span style="color:lightgrey">[obs]</span>', 
     'ylabel': 'DOY min. flow (julian day) <span style="color:lightgrey">[sim]</span>'},
    {'x': 'intensity_obs', 'y': 'intensity_sim', 'range': [-0.1, 1.2], 'dtick': 0.3, 'row': 2, 'col': 2,
     'xlabel': 'Drought intensity (-) <span style="color:lightgrey">[obs]</span>', 
     'ylabel': 'Drought intensity (-) <span style="color:lightgrey">[sim]</span>'},
    {'x': 'severity_obs', 'y': 'severity_sim', 'range': [-100, 1570], 'dtick': 500, 'row': 1, 'col': 3,
     'xlabel': 'Drought severity (mm) <span style="color:lightgrey">[obs]</span>', 
     'ylabel': 'Drought severity (mm) <span style="color:lightgrey">[sim]</span>'},
    {'x': 'events_per_year_obs', 'y': 'events_per_year_sim', 'range': [0, 4], 'dtick': 1, 'row': 2, 'col': 3,
     'xlabel': 'Annual drought frequency (-) <span style="color:lightgrey">[obs]</span>', 
     'ylabel': 'Annual drought frequency (-) <span style="color:lightgrey">[sim]</span>'},
]

# Create all plots
for i, plot in enumerate(plots):
    add_reference_line(plot['range'], plot['row'], plot['col'])
    add_scatter_plot(plot['x'], plot['y'], 'train', plot['row'], plot['col'])
    add_scatter_plot(plot['x'], plot['y'], 'validation', plot['row'], plot['col'])
    add_scatter_plot(plot['x'], plot['y'], 'test', plot['row'], plot['col'], 
                    show_colorbar=(i == 0))  # Only show colorbar on first plot
    
    fig.update_xaxes(title_text=plot['xlabel'], range=plot['range'],
                    dtick=plot['dtick'], row=plot['row'], col=plot['col'])
    fig.update_yaxes(title_text=plot['ylabel'], range=plot['range'],
                    dtick=plot['dtick'], row=plot['row'], col=plot['col'])

# Calculate Spearman correlation for each plot (train, validation, test separately)
correlations = []
for plot in plots:
    r_train, _      = spearmanr(andeanGC_metadata[andeanGC_metadata.split == 'train'][plot['x']],
                                andeanGC_metadata[andeanGC_metadata.split == 'train'][plot['y']])
    r_validation, _ = spearmanr(andeanGC_metadata[andeanGC_metadata.split == 'validation'][plot['x']],
                                andeanGC_metadata[andeanGC_metadata.split == 'validation'][plot['y']])
    r_test, _       = spearmanr(andeanGC_metadata[andeanGC_metadata.split == 'test'][plot['x']],
                                andeanGC_metadata[andeanGC_metadata.split == 'test'][plot['y']])
    correlations.append({'train': r_train, 'validation': r_validation, 'test': r_test})

# Annotation positions: (paper_x, paper_y) for each subplot
subplot_positions = [
    (0.19, 0.60),  # row=1, col=1
    (0.59, 0.60),  # row=1, col=2
    (0.99, 0.60),  # row=1, col=3
    (0.19, 0.01),   # row=2, col=1
    (0.59, 0.01),   # row=2, col=2
    (0.99, 0.01),   # row=2, col=3
]

annotations = [
    dict(
        x=0.01, y=0.99, xref="paper", yref="paper",
        text="<span style='color:lightgrey'>○</span> Train (Tr)<br>"
             "<span style='color:black'>○</span> Val (Va)<br>"
             "<span style='color:black'>○</span> Test (Ts)<br>",
        showarrow=False, font=dict(size=16), bgcolor="rgba(255,255,255,0.55)",
        borderwidth=0, align="left"
    )
]
for i, (px, py) in enumerate(subplot_positions):
    r = correlations[i]
    annotations.append(dict(
        x=px, y=py, xref="paper", yref="paper",
        text=(f"r = {r['train']:.2f} (Tr)<br>"
              f"r = {r['validation']:.2f} (Va)<br>"
              f"r = {r['test']:.2f} (Ts)"),
        showarrow=False, font=dict(size=13), bgcolor="rgba(255,255,255,0.8)", borderwidth=0,
        align="left"
    ))

# Panel labels (a-f) in the upper right corner of each subplot, in reading order (row-major)
panel_labels = ['a)', 'b)', 'c)', 'd)', 'e)', 'f)']
panel_x = [0.27, 0.62, 0.98]  # right edge of each column
panel_y = [0.995, 0.445]       # top edge of each row
plots_reading_order = sorted(plots, key=lambda p: (p['row'], p['col']))
for label, plot in zip(panel_labels, plots_reading_order):
    annotations.append(dict(
        x=panel_x[plot['col'] - 1], y=panel_y[plot['row'] - 1], xref="paper", yref="paper",
        text=f"<b>{label}</b>", showarrow=False, font=dict(size=16, family=FONT_FAMILY, color=PANEL_LABEL_COLOR),
        xanchor="right", yanchor="top"
    ))

# Update layout
fig.update_xaxes(title_standoff=2)
fig.update_yaxes(title_standoff=2)
fig.update_layout(**base_layout(
    height=650, width=1000, font=dict(size=14.5), annotations=annotations,
    margin=dict(l=60, r=20, b=50, t=20)))

save_figure(fig, "figures/fig_performance_droughts.svg", height=650, width=1100)
fig.show()
